# Full manual lineage: Hospital readmissions

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/9n3s-kdb3. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "04_hospital_readmissions"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/9n3s-kdb3"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'57f3e6def441dc76': 'd5dd538630f79241f3ffbe110bca838b3b20a51d8ffd2251a3382e2c3b8b76bb', 'e762fd9e10d69c84': '7e7915dd2751a1e59ac46dbc9194fb781e820962a04d47c31344ea2bb30c91f6'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Volume-aware uncertainty context alongside, not in place of, CMS adjusted ERR."""


import numpy as np
import pandas as pd
from scipy.stats import beta


def observed_rate_review(frame: pd.DataFrame, *, prior_discharges: float = 200) -> pd.DataFrame:
    """Beta-binomial smoothing of observed rates with state/measure pooled priors.

    This is an exchangeability assumption, not case-mix adjustment. Credible
    intervals condition on an estimated peer mean and fixed prior strength;
    they do not quantify uncertainty in the publisher's adjusted ERR.
    """
    if prior_discharges <= 0:
        raise ValueError("Prior discharge mass must be positive")
    x = frame.copy()
    if x.duplicated(["facility_id", "measure_name"]).any():
        raise ValueError("Facility/measure grain is not unique")
    n = pd.to_numeric(x.number_of_discharges, errors="coerce")
    events = pd.to_numeric(x.number_of_readmissions, errors="coerce")
    valid = n.gt(0) & events.ge(0) & events.le(n)
    x["usable_observed_count_flag"] = valid.astype(int)
    x["usable_n"] = n.where(valid)
    x["usable_events"] = events.where(valid)
    group = x.groupby(["state", "measure_name"], dropna=False)
    peer_n = group.usable_n.transform("sum") - x.usable_n
    peer_events = group.usable_events.transform("sum") - x.usable_events
    # Leave this facility out of its prior mean to avoid using its events twice.
    mean = (peer_events / peer_n.where(peer_n > 0)).where(
        peer_events.gt(0) & peer_events.lt(peer_n)
    )
    a = prior_discharges * mean + events
    b = prior_discharges * (1 - mean) + n - events
    x["observed_rate"] = events / n.where(valid)
    x["peer_observed_rate"] = mean
    x["reliability_weight_observed"] = (n / (n + prior_discharges)).where(valid & mean.notna())
    x["posterior_observed_rate"] = (a / (a + b)).where(valid & mean.notna())
    x["observed_rate_credible_low"] = beta.ppf(0.025, a, b)
    x["observed_rate_credible_high"] = beta.ppf(0.975, a, b)
    x.loc[~valid | mean.isna(), ["observed_rate_credible_low", "observed_rate_credible_high"]] = (
        np.nan
    )
    x["observed_rate_gap_pp"] = 100 * (x.posterior_observed_rate - mean)
    x["prior_discharges"] = prior_discharges
    return x.drop(columns=["usable_n", "usable_events"])


def shrinkage_sensitivity(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for mass in [50, 200, 500]:
        x = observed_rate_review(frame, prior_discharges=mass)
        rows.append(
            {
                "prior_discharges": mass,
                "estimable_records": int(x.posterior_observed_rate.notna().sum()),
                "above_peer_lower_bound": int(
                    x.observed_rate_credible_low.gt(x.peer_observed_rate).sum()
                ),
                "median_credible_width_pp": 100
                * (x.observed_rate_credible_high - x.observed_rate_credible_low).median(),
            }
        )
    return pd.DataFrame(rows)

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def _cms() -> pd.DataFrame:

    meta_url = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/9n3s-kdb3"
    metadata = _get(meta_url).json()
    distributions = metadata.get("distribution", [])
    csv_url = next(
        (
            d.get("downloadURL")
            for d in distributions
            if "csv"
            in str(d.get("format") or d.get("mediaType") or d.get("description") or "").lower()
        ),
        None,
    )
    if not csv_url:
        raise RuntimeError("CMS metadata did not expose a CSV distribution")
    response = _get(csv_url)
    preserve_response(
        response,
        project_dir("04_hospital_readmissions") / "data" / "raw",
        filename="hrrp.csv",
        publisher="Centers for Medicare & Medicaid Services",
        source_url=meta_url,
        coverage="See publisher reporting-period fields in preserved CSV",
        grain="Facility and readmission measure",
        refresh_frequency="Annual with publisher revisions",
        reuse_basis="Public CMS provider data; retain suppression and methodology caveats",
        attribution="CMS Hospital Readmissions Reduction Program",
        limitations="Suppressed measures and counts; observational, risk-adjusted metrics are not causal effects",
    )
    raw = pd.read_csv(BytesIO(response.content), dtype={"Facility ID": "string"}, low_memory=False)
    return raw.rename(
        columns={
            "Facility ID": "facility_id",
            "Facility Name": "facility_name",
            "State": "state",
            "Measure Name": "measure_name",
            "Number of Discharges": "number_of_discharges",
            "Number of Readmissions": "number_of_readmissions",
            "Excess Readmission Ratio": "excess_readmission_ratio",
        }
    )


Project: projects/hospital_readmissions
Publisher: https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/9n3s-kdb3
Pinned response folder: projects/hospital_readmissions/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
raw = _cms()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 2
Imported rows: 18330 columns: 12


,facility_name,facility_id,state,measure_name,number_of_discharges,Footnote,excess_readmission_ratio,Predicted Readmission Rate,Expected Readmission Rate,number_of_readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-CABG-HRRP,137.0,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-AMI-HRRP,273.0,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024


## 3 Transform and clean

Run the original Hospital readmissions transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def readmissions(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(
        x, ["facility_id", "facility_name", "state", "measure_name", "excess_readmission_ratio"]
    )
    x["excess_readmission_ratio"] = pd.to_numeric(x["excess_readmission_ratio"], errors="coerce")
    x["number_of_readmissions"] = pd.to_numeric(x.get("number_of_readmissions"), errors="coerce")
    x["number_of_discharges"] = pd.to_numeric(x.get("number_of_discharges"), errors="coerce")
    rural = x["rural_flag"] if "rural_flag" in x else pd.Series(np.nan, index=x.index)
    svi = x["svi_percentile"] if "svi_percentile" in x else pd.Series(np.nan, index=x.index)
    x["rural_flag"] = pd.to_numeric(rural, errors="coerce").astype("Int64")
    x["svi_percentile"] = pd.to_numeric(svi, errors="coerce")
    x["reliability_weight"] = (x["number_of_discharges"] / (x["number_of_discharges"] + 200)).round(
        4
    )
    x["shrunk_readmission_ratio"] = (
        1 + x["reliability_weight"] * (x["excess_readmission_ratio"] - 1)
    ).round(4)
    x = observed_rate_review(x)
    # ERR is a ratio, not a probability: multiplying (ERR-1) by discharges
    # does not estimate excess events. Use the observed-rate gap separately.
    x["peer_gap_pp"] = x["observed_rate_gap_pp"]
    x["excess_readmission_opportunity"] = (
        x["peer_gap_pp"].clip(lower=0) / 100 * x["number_of_discharges"]
    )
    x["priority_score"] = (x["observed_rate_credible_low"] - x["peer_observed_rate"]).clip(
        lower=0
    ) * x["number_of_discharges"]
    x["performance_band"] = np.select(
        [x["excess_readmission_ratio"] > 1.05, x["excess_readmission_ratio"] < 0.95],
        ["Needs attention", "Better than expected"],
        default="Near expected",
    )
    x.loc[x.excess_readmission_ratio.isna(), "performance_band"] = "Not reported"
    return x[
        [
            "facility_id",
            "facility_name",
            "state",
            "measure_name",
            "number_of_discharges",
            "number_of_readmissions",
            "excess_readmission_ratio",
            "performance_band",
            "rural_flag",
            "svi_percentile",
            "reliability_weight",
            "shrunk_readmission_ratio",
            "excess_readmission_opportunity",
            "peer_gap_pp",
            "priority_score",
            "observed_rate",
            "peer_observed_rate",
            "posterior_observed_rate",
            "observed_rate_credible_low",
            "observed_rate_credible_high",
        ]
    ]

clean = readmissions(raw)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 18330 columns: 20


,facility_id,facility_name,state,measure_name,number_of_discharges,number_of_readmissions,excess_readmission_ratio,performance_band,rural_flag,svi_percentile,reliability_weight,shrunk_readmission_ratio,excess_readmission_opportunity,peer_gap_pp,priority_score,observed_rate,peer_observed_rate,posterior_observed_rate,observed_rate_credible_low,observed_rate_credible_high
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,Near expected,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-CABG-HRRP,137.0,13.0,0.9531,Near expected,<NA>,NaN,0.4065,0.9809,0.0,-1.775847,0.0,0.094891,0.138574,0.120815,0.088270,0.157619
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-AMI-HRRP,273.0,33.0,0.9370,Better than expected,<NA>,NaN,0.5772,0.9636,0.0,-1.432658,0.0,0.120879,0.145701,0.131375,0.102479,0.163221


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the observed_rate_uncertainty output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)


def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["observed_rate_uncertainty"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 6, 'bootstrap_gap': 1, 'feature_diagnostics': 12, 'temporal_diagnostics': 0, 'threshold_sensitivity': 5, 'decision_queue': 20, 'observed_rate_uncertainty': 18330, 'shrinkage_sensitivity': 3}


,facility_id,facility_name,state,measure_name,number_of_discharges,number_of_readmissions,excess_readmission_ratio,performance_band,rural_flag,svi_percentile,...,priority_score,observed_rate,peer_observed_rate,posterior_observed_rate,observed_rate_credible_low,observed_rate_credible_high,usable_observed_count_flag,reliability_weight_observed,observed_rate_gap_pp,prior_discharges
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,Near expected,<NA>,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,200
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-CABG-HRRP,137.0,13.0,0.9531,Near expected,<NA>,NaN,...,0.0,0.094891,0.138574,0.120815,0.088270,0.157619,1,0.406528,-1.775847,200
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-AMI-HRRP,273.0,33.0,0.9370,Better than expected,<NA>,NaN,...,0.0,0.120879,0.145701,0.131375,0.102479,0.163221,1,0.577167,-1.432658,200
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-COPD-HRRP,122.0,19.0,0.9823,Near expected,<NA>,NaN,...,0.0,0.155738,0.184783,0.173778,0.134433,0.216959,1,0.378882,-1.100459,200
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-PN-HRRP,507.0,79.0,0.9871,Near expected,<NA>,NaN,...,0.0,0.155819,0.155231,0.155652,0.129891,0.183258,1,0.717115,0.042156,200


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
KEY_COLUMNS = ["facility_id","measure_name"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 18330 clean rows: 18330
Quality: {'rows': 18330, 'columns': 20, 'duplicate_rows': 0, 'missing_cells': 166481, 'missing_rate': 0.454122, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after the checks pass. It writes the dated extract, quality report, analytical tables, and source evidence under `data/manual_rebuild/`. If a fresh response has the same date as an existing rebuild, Save uses a source-digest suffix so the verified result is not overwritten; it refuses a conflicting partial folder.

**Check before continuing:** every named output exists in the dated folder; the source digest and reported row counts match the current response.


In [6]:
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    prior_file = OUTPUT_DIR / "quality_report.json"
    prior_hashes = json.loads(prior_file.read_text(encoding="utf-8")).get("source_sha256") if prior_file.is_file() else None
    if isinstance(prior_hashes, str):
        prior_hashes = [prior_hashes]
    if prior_hashes != CURRENT_SOURCE_SHA:
        digest_suffix = hashlib.sha256(json.dumps(CURRENT_SOURCE_SHA).encode("utf-8")).hexdigest()[:12]
        OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / f"{SNAPSHOT_DATE}-{digest_suffix}"
        if OUTPUT_DIR.exists():
            raise FileExistsError(f"A different or partial rebuild already exists at {OUTPUT_DIR}; review it before saving")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "CMS Hospital Readmissions Reduction Program",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


Saved manual rebuild to: projects/hospital_readmissions/data/manual_rebuild/2026-09-26


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `schema.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `populate.sql`, then run `validation.sql` and `analysis.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [7]:
display(analysis_outputs["observed_rate_uncertainty"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,facility_id,facility_name,state,measure_name,number_of_discharges,number_of_readmissions,excess_readmission_ratio,performance_band,rural_flag,svi_percentile,...,priority_score,observed_rate,peer_observed_rate,posterior_observed_rate,observed_rate_credible_low,observed_rate_credible_high,usable_observed_count_flag,reliability_weight_observed,observed_rate_gap_pp,prior_discharges
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,Near expected,<NA>,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,200
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-CABG-HRRP,137.0,13.0,0.9531,Near expected,<NA>,NaN,...,0.0,0.094891,0.138574,0.120815,0.088270,0.157619,1,0.406528,-1.775847,200
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-AMI-HRRP,273.0,33.0,0.9370,Better than expected,<NA>,NaN,...,0.0,0.120879,0.145701,0.131375,0.102479,0.163221,1,0.577167,-1.432658,200
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-COPD-HRRP,122.0,19.0,0.9823,Near expected,<NA>,NaN,...,0.0,0.155738,0.184783,0.173778,0.134433,0.216959,1,0.378882,-1.100459,200
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-PN-HRRP,507.0,79.0,0.9871,Near expected,<NA>,NaN,...,0.0,0.155819,0.155231,0.155652,0.129891,0.183258,1,0.717115,0.042156,200
5,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,READM-30-HF-HRRP,653.0,136.0,1.0233,Near expected,<NA>,NaN,...,0.0,0.208270,0.200558,0.206461,0.179979,0.234248,1,0.765533,0.590335,200
6,010005,MARSHALL MEDICAL CENTERS,AL,READM-30-CABG-HRRP,NaN,NaN,NaN,Not reported,<NA>,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,200
7,010005,MARSHALL MEDICAL CENTERS,AL,READM-30-AMI-HRRP,NaN,NaN,NaN,Not reported,<NA>,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,200
8,010005,MARSHALL MEDICAL CENTERS,AL,READM-30-COPD-HRRP,132.0,17.0,0.9308,Better than expected,<NA>,NaN,...,0.0,0.128788,0.185886,0.163185,0.125475,0.204735,1,0.397590,-2.270182,200
9,010005,MARSHALL MEDICAL CENTERS,AL,READM-30-HF-HRRP,164.0,35.0,1.0087,Near expected,<NA>,NaN,...,0.0,0.213415,0.200810,0.206489,0.166535,0.249498,1,0.450549,0.567879,200


Tableau input: projects/hospital_readmissions/data/manual_rebuild/2026-09-26/tableau_extract.csv


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [8]:
DECISION_TEXT = "For the 2026-09-26 CMS HRRP snapshot, prioritize manual evidence review of 232 lower-bound signals among 8,022 scored facility-condition records. Verify each case in analysis/observed_rate_uncertainty.csv; the packaged Tableau evidence-review worksheet shows the saved ten-case queue."
INTERPRETATION_LIMIT = "This is an unadjusted observed-rate screening comparison, not evidence of preventable admissions, savings, or care quality. Small denominators, suppressed or unsupported pairs, case mix, peer-rate uncertainty, and multiple comparisons constrain interpretation."
EVIDENCE_FILES = ["quality_report.json", "analysis/observed_rate_uncertainty.csv"]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)


Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/observed_rate_uncertainty.csv']
Decision: For the 2026-09-26 CMS HRRP snapshot, prioritize manual evidence review of 232 lower-bound signals among 8,022 scored facility-condition records. Verify each case in analysis/observed_rate_uncertainty.csv; the packaged Tableau evidence-review worksheet shows the saved ten-case queue.
Limit: This is an unadjusted observed-rate screening comparison, not evidence of preventable admissions, savings, or care quality. Small denominators, suppressed or unsupported pairs, case mix, peer-rate uncertainty, and multiple comparisons constrain interpretation.
